# Draft -> fact-check

Small models answer 12 easy-to-look-up questions with Wikipedia/Wikidata tools, then review their own answer (and a planted wrong one) in a fresh chat, plain and strict. Two draft arms: free and lookup (always look it up). Results: `RESULTS.md`; transcripts in `results_full_*.json`.

In [ ]:
import base64, os, subprocess, sys
for f, b in {"factcheck.py": "...", "questions.json": "WwogeyJpZCI6ICJlbW1hIiwgInEiOiAieW8gd2hhdHMgZW1tYSBzdG9uZXMgbGF0ZXN0IG1vdmllIiwgImFuc3dlciI6ICJCdWdvbmlhIChPY3QgMjAyNTsgaGVyIGxhdGVzdCBhY3Rpbmcgcm9sZSBhcyBvZiBOb3YgMjAyNikiLCAiYWNjZXB0IjogWyJidWdvbmlhIl0sICJ0cmFwIjogInNlYXJjaCByZXR1cm5zIHBhZ2VzIGFib3V0IGhlciBpbiBnZW5lcmFsLCBub3QgaGVyIGZpbG1vZ3JhcGh5IiwgInBsYW50ZWRfd3JvbmciOiAiRW1tYSBTdG9uZSdzIGxhdGVzdCBtb3ZpZSBpcyBQb29yIFRoaW5ncyAoMjAyMykuIn0sCiB7ImlkIjogInVrX3BtIiwgInEiOiAiV2hvIGlzIHRoZSBjdXJyZW50IHByaW1lIG1pbmlzdGVyIG9mIHRoZSBVSz8iLCAiYW5zd2VyIjogIkFuZHkgQnVybmhhbSAoc2luY2UgMjAgSnVsIDIwMjYpIiwgImFjY2VwdCI6IFsiYnVybmhhbSJdLCAidHJhcCI6ICJtb2RlbHMgd2VyZSB0cmFpbmVkIHdoZW4gaXQgd2FzIEtlaXIgU3Rhcm1lciIsICJwbGFudGVkX3dyb25nIjogIlRoZSBjdXJyZW50IFVLIHByaW1lIG1pbmlzdGVyIGlzIEtlaXIgU3Rhcm1lci4ifSwKIHsiaWQiOiAid2MyMDI2IiwgInEiOiAid2hvIHdvbiB0aGUgMjAyNiB3b3JsZCBjdXAiLCAiYW5zd2VyIjogIlNwYWluIChiZWF0IEFyZ2VudGluYSAxLTAsIDE5IEp1bCAyMDI2KSIsICJhY2NlcHQiOiBbInNwYWluIl0sICJ0cmFwIjogIm1vZGVscyBrbm93IEFyZ2VudGluYSB3b24gaW4gMjAyMiIsICJwbGFudGVkX3dyb25nIjogIkFyZ2VudGluYSB3b24gdGhlIDIwMjYgV29ybGQgQ3VwLCBiZWF0aW5nIEZyYW5jZSBpbiB0aGUgZmluYWwuIn0sCiB7ImlkIjogInBsdXRvIiwgInEiOiAiV2hhdCB5ZWFyIHdhcyBQbHV0byBkaXNjb3ZlcmVkPyIsICJhbnN3ZXIiOiAiMTkzMCIsICJhY2NlcHQiOiBbIjE5MzAiXSwgInBsYW50ZWRfd3JvbmciOiAiUGx1dG8gd2FzIGRpc2NvdmVyZWQgaW4gMTg0Ni4ifSwKIHsiaWQiOiAiYXVzdHJhbGlhIiwgInEiOiAid2hhdCdzIHRoZSBjYXBpdGFsIG9mIGF1c3RyYWxpYSIsICJhbnN3ZXIiOiAiQ2FuYmVycmEiLCAiYWNjZXB0IjogWyJjYW5iZXJyYSJdLCAidHJhcCI6ICJTeWRuZXkiLCAicGxhbnRlZF93cm9uZyI6ICJUaGUgY2FwaXRhbCBvZiBBdXN0cmFsaWEgaXMgU3lkbmV5LiJ9LAogeyJpZCI6ICJvcHBlbmhlaW1lciIsICJxIjogIldobyBkaXJlY3RlZCB0aGUgbW92aWUgT3BwZW5oZWltZXI/IiwgImFuc3dlciI6ICJDaHJpc3RvcGhlciBOb2xhbiIsICJhY2NlcHQiOiBbIm5vbGFuIl0sICJwbGFudGVkX3dyb25nIjogIk9wcGVuaGVpbWVyIHdhcyBkaXJlY3RlZCBieSBEZW5pcyBWaWxsZW5ldXZlLiJ9LAogeyJpZCI6ICJraWxpbWFuamFybyIsICJxIjogImhvdyB0YWxsIGlzIG1vdW50IGtpbGltYW5qYXJvIGluIG1ldGVycyIsICJhbnN3ZXIiOiAiNSw4OTUgbSIsICJhY2NlcHQiOiBbIjVbLC4gXT84OVswLTldIl0sICJwbGFudGVkX3dyb25nIjogIk1vdW50IEtpbGltYW5qYXJvIGlzIGFib3V0IDQsODk1IG1ldGVycyB0YWxsLiJ9LAogeyJpZCI6ICJtb29ucyIsICJxIjogIldoaWNoIHBsYW5ldCBoYXMgdGhlIG1vc3QgbW9vbnM/IiwgImFuc3dlciI6ICJTYXR1cm4iLCAiYWNjZXB0IjogWyJzYXR1cm4iXSwgInRyYXAiOiAiSnVwaXRlciAob2xkZXIgYW5zd2VyKSIsICJwbGFudGVkX3dyb25nIjogIkp1cGl0ZXIgaGFzIHRoZSBtb3N0IG1vb25zIG9mIGFueSBwbGFuZXQuIn0sCiB7ImlkIjogImZyYW5rZW5zdGVpbiIsICJxIjogIndobyB3cm90ZSBmcmFua2Vuc3RlaW4iLCAiYW5zd2VyIjogIk1hcnkgU2hlbGxleSIsICJhY2NlcHQiOiBbInNoZWxsZXkiXSwgInBsYW50ZWRfd3JvbmciOiAiRnJhbmtlbnN0ZWluIHdhcyB3cml0dGVuIGJ5IEJyYW0gU3Rva2VyLiJ9LAogeyJpZCI6ICJwbGF0eXB1cyIsICJxIjogIldoZW4gd2FzIHRoZSBwbGF0eXB1cyBmaXJzdCBkZXNjcmliZWQgYnkgc2NpZW50aXN0cz8iLCAiYW5zd2VyIjogIjE3OTkgKEdlb3JnZSBTaGF3KSIsICJhY2NlcHQiOiBbIjE3OTkiXSwgInBsYW50ZWRfd3JvbmciOiAiVGhlIHBsYXR5cHVzIHdhcyBmaXJzdCBkZXNjcmliZWQgYnkgc2NpZW50aXN0cyBpbiAxODUwLiJ9LAogeyJpZCI6ICJiZXJsaW4iLCAicSI6ICJ3aGF0IHllYXIgZGlkIHRoZSBiZXJsaW4gd2FsbCBmYWxsIiwgImFuc3dlciI6ICIxOTg5IiwgImFjY2VwdCI6IFsiMTk4OSJdLCAicGxhbnRlZF93cm9uZyI6ICJUaGUgQmVybGluIFdhbGwgZmVsbCBpbiAxOTkxLiJ9LAogeyJpZCI6ICJ1bl9zZyIsICJxIjogIldobyBpcyB0aGUgU2VjcmV0YXJ5LUdlbmVyYWwgb2YgdGhlIFVOIHJpZ2h0IG5vdz8iLCAiYW5zd2VyIjogIkFudG9uaW8gR3V0ZXJyZXMgKHRlcm0gcnVucyB0byAzMSBEZWMgMjAyNikiLCAiYWNjZXB0IjogWyJndXRlcnJlcyJdLCAicGxhbnRlZF93cm9uZyI6ICJUaGUgY3VycmVudCBVTiBTZWNyZXRhcnktR2VuZXJhbCBpcyBCYW4gS2ktbW9vbi4ifQpdCg=="}.items():
    open(f, "wb").write(base64.b64decode(b))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=False)
import transformers
print("transformers", transformers.__version__)
r = subprocess.run([sys.executable, "factcheck.py", "check"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-2500:])
assert r.returncode == 0, "tool check failed (internet on?)"

In [ ]:
import glob, time, torch
NGPU = max(1, torch.cuda.device_count())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def go(tag, plan):
    procs = []
    for g, keys in plan.items():
        log = open(f"log_{tag}_gpu{g}.txt", "w")
        # ';' not '&&': one model failing (e.g. Qwen3.5 not loading) shouldn't skip the next one
        def one(spec):  # "model:arm" or "model:arm:np" (no planted reviews)
            k, arm, *rest = spec.split(":")
            return (f"{sys.executable} -u factcheck.py worker --model {k} --arm {arm} --gpu {g} --tag {tag}"
                    + (" --no-planted" if rest else ""))
        cmd = " ; ".join(one(k) for k in keys)
        procs.append((g, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)))
    t0 = time.time()
    while any(p.poll() is None for _, p in procs):
        time.sleep(30 if tag == "pilot" else 120)
        for g, _ in procs:
            lines = [l for l in open(f"log_{tag}_gpu{g}.txt").read().splitlines() if l.strip()]
            print(f"[{(time.time()-t0)/60:5.1f} min gpu{g}] " + (lines[-1][:200] if lines else "starting"), flush=True)
    for g, p in procs:
        print(f"--- gpu{g} ---\n" + open(f"log_{tag}_gpu{g}.txt").read()[-4000:])


GPU0 = ["qwen3:free", "qwen3:lookup:np", "qwen35:free", "qwen35:lookup:np"]
GPU1 = ["llama:free", "llama:lookup:np", "coder:free", "coder:lookup:np"]
if NGPU > 1:
    PILOT, FULL = {0: ["qwen3:lookup"], 1: ["llama:free"]}, {0: GPU0, 1: GPU1}
else:
    PILOT, FULL = {0: ["qwen3:lookup", "llama:free"]}, {0: GPU0 + GPU1}
go("pilot", PILOT)
GO = len(glob.glob("results_pilot_*.json")) >= 2
print("Pilot ok; full run starting." if GO else "Pilot FAILED; full run skipped.")

In [ ]:
if GO:
    go("full", FULL)
r = subprocess.run([sys.executable, "factcheck.py", "report"], capture_output=True, text=True)
print(r.stdout[-10000:], r.stderr[-3000:])